# Day 2 — EDA + 결합 데이터 (Colab 시연 노트북)

이 노트북은 강의 흐름을 그대로 따라가는 코드를 미리 채워둔 시연·연습용입니다.
- Claude Code가 짠 코드를 옮겨 직접 만지는 자리
- 데이터는 강의 레포(team-datapopcorn/inflearn_ccd_challenge)에서 받거나 출처에서 직접
- 셀 단위로 ▶︎ 실행 → 변수·임계값 바꿔보기

## 사전 준비
좌측 폴더 아이콘 → 다음 CSV 업로드:
- `기간별_일평균_대기환경_정보_2024년.csv` (서울 미세먼지 — 광장 OA-2218)
- `기간별_일평균_대기환경_정보_2025년.csv`
- `seoul_bike_daily_2024.csv` (광장 따릉이 일별)
- `seoul_subway_daily_total_2024.csv` (공공데이터포털 → 일별 합계로 가공)
- `OBS_ASOS_DD_20260510160758.csv` (기상청 ASOS — 받기 까다로워서 강의 레포 백업본 사용)
- `time_series_KR_20240101-0000_20260510-1610.csv` (구글 트렌드 "미세먼지" 월별)


In [ ]:
# ── 한글 폰트 (Colab) ────────────────────────────────────
!apt -qq install fonts-nanum > /dev/null
import matplotlib as mpl, matplotlib.pyplot as plt, matplotlib.font_manager as fm
fm._load_fontmanager(try_read_cache=False)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
print('한글 폰트 OK')

## 1단계 — 미세먼지 통합 EDA

서울 미세먼지 2024 + 2025 통합. shape · 결측 · 자치구 · 시즌성 · spike · 등급 분포.


In [ ]:
import pandas as pd
import numpy as np

a = pd.read_csv('기간별_일평균_대기환경_정보_2024년.csv', encoding='cp949')
b = pd.read_csv('기간별_일평균_대기환경_정보_2025년.csv', encoding='cp949')

a = a.rename(columns={'측정일시':'date', '측정소명':'district',
                      '미세먼지(㎍/㎥)':'pm10', '초미세먼지(㎍/㎥)':'pm25'})[['date','district','pm10','pm25']]
b = b.rename(columns={'측정일시':'date', '측정소명':'district',
                      '미세먼지농도(㎍/㎥)':'pm10', '초미세먼지농도(㎍/㎥)':'pm25'})[['date','district','pm10','pm25']]

air = pd.concat([a, b], ignore_index=True)
air['date'] = pd.to_datetime(air['date'].astype(str), format='%Y%m%d')
air['year'] = air['date'].dt.year
air['month'] = air['date'].dt.month
print('shape:', air.shape, '· 측정소:', air['district'].nunique())
print('기간:', air['date'].min().date(), '~', air['date'].max().date())
print('PM2.5 결측:', f"{air['pm25'].isna().mean()*100:.2f}%")
air.head()

In [ ]:
# 자치구별 PM2.5 연평균 — 2024 vs 2025
g = air.groupby(['district','year'])['pm25'].mean().unstack('year').sort_values(2025, ascending=False)
fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(g)); w = 0.4
ax.bar(x-w/2, g[2024], w, label='2024', color='#94A3B8')
ax.bar(x+w/2, g[2025], w, label='2025', color='#4A7BF7')
ax.set_xticks(x); ax.set_xticklabels(g.index, rotation=45, ha='right')
ax.set_ylabel('PM2.5 연평균 (㎍/㎥)')
ax.set_title('자치구별 PM2.5 연평균 — 2024 vs 2025')
ax.legend(); ax.grid(axis='y', alpha=0.25, linestyle='--')
plt.tight_layout(); plt.show()

In [ ]:
# 월별 시즌성 + 황사 시즌 (3~5월) 음영
monthly = air.groupby(['year','month'])['pm25'].mean().unstack('year')
fig, ax = plt.subplots(figsize=(11, 5))
ax.axvspan(2.5, 5.5, color='#FBBF24', alpha=0.15, label='황사 시즌')
ax.plot(monthly.index, monthly[2024], marker='o', color='#94A3B8', label='2024')
ax.plot(monthly.index, monthly[2025], marker='o', color='#4A7BF7', label='2025', linewidth=2)
ax.set_xticks(range(1,13)); ax.set_xticklabels([f'{m}월' for m in range(1,13)])
ax.set_ylabel('PM2.5 월평균 (㎍/㎥)')
ax.set_title('월별 PM2.5 — 2024 vs 2025 (+ 황사 시즌)')
ax.legend(); ax.grid(alpha=0.25, linestyle='--')
plt.tight_layout(); plt.show()

## 2단계 — 따릉이·지하철 결합 (2024년 매칭)

> 따릉이/지하철 raw는 2024년까지만 공개됨 (2025년은 광장에 미등록).
> 결합 분석은 2024년 1년치로 한정.

미세먼지 등급(좋음/보통/나쁨/매우나쁨)별 두 교통수단 이용량 비교.


In [ ]:
bike = pd.read_csv('seoul_bike_daily_2024.csv', encoding='cp949')
bike = bike.rename(columns={'대여일자':'date', '대여건수':'rides'})
bike['date'] = pd.to_datetime(bike['date'])

sub = pd.read_csv('seoul_subway_daily_total_2024.csv', encoding='cp949')
sub = sub.rename(columns={'수송일자':'date', '총승하차인원':'passengers'})[['date','passengers']]
sub['date'] = pd.to_datetime(sub['date'])

# 2024년 일별 PM2.5 평균 (자치구 평균)
daily_pm = (air[air['year']==2024].groupby('date')['pm25'].mean().reset_index())
daily_pm['grade'] = pd.cut(daily_pm['pm25'], bins=[-1, 15, 35, 75, 1000],
                            labels=['좋음','보통','나쁨','매우나쁨'])

bike_d = bike.merge(daily_pm[['date','grade']], on='date')
sub_d = sub.merge(daily_pm[['date','grade']], on='date')

bike_avg = bike_d.groupby('grade', observed=True)['rides'].mean().reindex(['좋음','보통','나쁨','매우나쁨'])
sub_avg = sub_d.groupby('grade', observed=True)['passengers'].mean().reindex(['좋음','보통','나쁨','매우나쁨'])
print('따릉이 등급별 평균:'); print(bike_avg)
print('\n지하철 등급별 평균(M):'); print(sub_avg/1e6)

In [ ]:
# 따릉이 vs 지하철 — 환경 민감도 비교
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#10B981','#4A7BF7','#FFB020','#FF6B6B']

ax = axes[0]
ax.bar(bike_avg.index, bike_avg.values, color=colors)
ax.set_title('따릉이 — 환경에 민감')
ax.set_ylabel('일평균 대여 건수')
ax.grid(axis='y', alpha=0.25, linestyle='--')

ax = axes[1]
ax.bar(sub_avg.index, sub_avg.values, color=colors)
ax.set_title('지하철 — 환경에 둔감')
ax.set_ylabel('일평균 총 승하차')
ax.grid(axis='y', alpha=0.25, linestyle='--')

fig.suptitle('미세먼지 등급별 따릉이 vs 지하철 (2024)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 3단계 — 날씨 결합 (2024 + 2025)

기상청 ASOS 서울(108) 일자료. 강의 레포 `data/weather/OBS_ASOS_DD_*.csv`.

PM2.5 vs 풍속·강수·기온·습도 상관 + 풍속-PM 산점도 (시즌별).


In [ ]:
# 기상청 ASOS 서울 일자료 (CP949)
weather = pd.read_csv('OBS_ASOS_DD_20260510160758.csv', encoding='cp949')
weather = weather.rename(columns={
    '일시': 'date',
    '평균기온(°C)': 'temp_avg',
    '최저기온(°C)': 'temp_min',
    '최고기온(°C)': 'temp_max',
    '일강수량(mm)': 'rain',
    '평균 풍속(m/s)': 'wind_avg',
    '최다풍향(16방위)': 'wind_dir',
    '평균 상대습도(%)': 'humidity_avg',
})[['date','temp_avg','temp_min','temp_max','rain','wind_avg','wind_dir','humidity_avg']]
weather['date'] = pd.to_datetime(weather['date'])
print('shape:', weather.shape)
weather.head(3)

In [ ]:
# PM2.5 일평균과 결합
daily_pm_all = air.groupby('date')['pm25'].mean().reset_index()
j = daily_pm_all.merge(weather, on='date', how='inner')
j['season'] = j['date'].dt.month.map(lambda m: '황사 (3~5월)' if 3<=m<=5 else '비황사')

# 날씨 변수와 PM2.5 상관계수
corr_cols = ['temp_avg','temp_min','temp_max','rain','wind_avg','humidity_avg']
labels_ko = ['평균기온','최저기온','최고기온','일강수량','평균풍속','평균습도']
corr = j[['pm25']+corr_cols].corr()['pm25'].drop('pm25')
print('PM2.5 vs 날씨 변수 상관계수:')
for k, v in zip(labels_ko, corr.values):
    print(f'  {k}: {v:+.3f}')

In [ ]:
# 풍속-PM2.5 산점도 (시즌별 분리)
fig, ax = plt.subplots(figsize=(10, 5))
for label, sub_df in j.groupby('season'):
    ax.scatter(sub_df['wind_avg'], sub_df['pm25'], alpha=0.45, s=20, label=label)
ax.set_xlabel('평균풍속 (m/s)'); ax.set_ylabel('PM2.5 (㎍/㎥)')
ax.set_title('풍속 vs PM2.5 — 시즌별')
ax.legend(); ax.grid(alpha=0.25, linestyle='--')
plt.tight_layout(); plt.show()

## 4단계 — 검색량(buzz) 결합 — 월별

구글 트렌드 "미세먼지" 월별 검색 인덱스 (0~100 정규화).
PM2.5 월평균 vs 검색 인덱스 시계열 + lag 상관.


In [ ]:
# 구글 트렌드 buzz 데이터 (UTF-8, 컬럼: Time, "미세먼지")
buzz = pd.read_csv('time_series_KR_20240101-0000_20260510-1610.csv')
buzz.columns = ['date', 'buzz']
buzz['date'] = pd.to_datetime(buzz['date'])
print('shape:', buzz.shape)
print('기간:', buzz['date'].min().date(), '~', buzz['date'].max().date())
buzz.head(3)

In [ ]:
# 월 단위로 집계 후 결합
air_m = (air.assign(month_p=air['date'].dt.to_period('M'))
         .groupby('month_p')['pm25'].mean().reset_index())
air_m['date'] = air_m['month_p'].dt.to_timestamp()

buzz_m = buzz.assign(month_p=buzz['date'].dt.to_period('M')).copy()
buzz_m = buzz_m.groupby('month_p')['buzz'].mean().reset_index()
buzz_m['date'] = buzz_m['month_p'].dt.to_timestamp()

j_buzz = air_m.merge(buzz_m[['month_p','buzz']], on='month_p').sort_values('date')

# 시계열 — 이중 y축
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(j_buzz['date'], j_buzz['pm25'], color='#4A7BF7', marker='o', label='PM2.5 월평균', linewidth=2)
ax1.set_ylabel('PM2.5 (㎍/㎥)', color='#4A7BF7')
ax1.tick_params(axis='y', labelcolor='#4A7BF7')
ax2 = ax1.twinx()
ax2.plot(j_buzz['date'], j_buzz['buzz'], color='#FF6B6B', marker='s', label='"미세먼지" 검색량', linewidth=2, alpha=0.85)
ax2.set_ylabel('검색 인덱스', color='#FF6B6B')
ax2.tick_params(axis='y', labelcolor='#FF6B6B')
ax1.set_title('월별 PM2.5 vs "미세먼지" 검색량')
plt.tight_layout(); plt.show()

In [ ]:
# lag 상관 — 검색량이 PM 농도를 며칠/월 따라가는지
print('lag 상관계수 (월 단위):')
for lag in range(-3, 4):
    c = j_buzz['pm25'].corr(j_buzz['buzz'].shift(-lag))
    print(f'  lag {lag:+d}월: corr={c:+.3f}')

## 변형 실습 (5분)

본인이 1개 골라 변형:
- A. PM2.5 → PM10 으로 같은 분석
- B. 이상치 IQR 1.5배 → 3배
- C. 결합 차트 등급 임계값 변경 (좋음/보통 vs 나쁨/매우나쁨 2분류)
- D. 본인이 궁금한 변형


## CLAUDE.md 누적

작업 폴더의 CLAUDE.md에 발견 규칙을 자연어로 추가:

```
오늘 EDA에서 발견한 규칙을 CLAUDE.md에 추가해줘.
- 측정일시는 항상 datetime으로 파싱
- 결측 50% 이상 측정소는 제외
- 이상치 기준은 IQR 3배 (황사 spike 인정)
- 황사 시즌 = 3~5월
- 결합 데이터 날짜 컬럼은 'date'로 통일
- 시즌별 컬러 4분위 통일, y축 단위 명시
```
